# How Many People in the World Can Squat 315 / 405 / 495?
### Two-pool model: gym squatters + non-gym strength athletes

> The bench press is almost entirely a gym movement — you need a bench and a barbell.  
> The squat is different. It's a fundamental human pattern.  
> Military recruits, combat sports athletes, rugby players, construction workers —  
> many have the leg strength to squat heavy without ever touching a squat rack.  
> This model accounts for both pools.

---
**Method:**  
OPL data (3.9M records) → fit squat distribution → adjust for gym-goer gap  
→ add non-gym strength athlete pool (military, combat sports, team sports S&C)  
→ 10-region world model → 100,000 Monte Carlo trials  
→ denominator analysis: all humans / able adults / men only / women only

---
## ⚙️ Setup

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import lognorm as _lognorm
import warnings, requests, zipfile, io, os
warnings.filterwarnings('ignore')
np.random.seed(42)

BG    = '#0d0d0d'
PANEL = '#141414'
GRID  = '#252525'
WHITE = '#f0f0f0'
GRAY  = '#666666'
C315  = '#00b4ff'
C405  = '#ff2d2d'
C495  = '#ffd700'
MALE  = '#4fc3f7'
FEM   = '#f48fb1'
POOL_A_COLOR = '#4fc3f7'   # gym squatters
POOL_B_COLOR = '#ff9800'   # non-gym strength athletes

MILESTONES = [315, 405, 495]
M_COLORS   = [C315, C405, C495]
M_TAGS     = ['315 lbs', '405 lbs', '495 lbs']
M_LABELS   = ['315 lbs\n(3 plates)', '405 lbs\n(4 plates)', '495 lbs\n(5 plates)']
FRAC_MALE  = 0.504
WORLD_POP  = 8_200_000_000

# Denominator reference populations (UN 2024)
POP_ABLE_1670  = 4_674_000_000   # world 16-70
POP_MEN_1670   = 2_372_000_000   # men 16-70
POP_WOMEN_1670 = 2_302_000_000   # women 16-70

REEL_W, REEL_H = 6.75, 12.0
REEL_DPI = 160

def make_fig(w=13, h=6, cols=1):
    if cols == 1:
        fig, ax = plt.subplots(figsize=(w, h), facecolor=BG)
        _style(ax); return fig, ax
    fig, axes = plt.subplots(1, cols, figsize=(w, h), facecolor=BG)
    [_style(ax) for ax in axes]; return fig, axes

def _style(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=GRAY, labelsize=11)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.xaxis.label.set_color(WHITE)
    ax.yaxis.label.set_color(WHITE)
    ax.title.set_color(WHITE)
    ax.grid(color=GRID, lw=0.6, alpha=0.8)

print('Ready.')

---
## 📊 Chapter 1 — OPL Squat Data

In [ ]:
OPL_PATH = 'opl_data.csv'
if not os.path.exists(OPL_PATH):
    print('Downloading OPL dataset...')
    url = 'https://openpowerlifting.gitlab.io/opl-csv/files/openpowerlifting-latest.zip'
    r   = requests.get(url, stream=True, timeout=180)
    chunks = []
    for chunk in r.iter_content(chunk_size=1024*256): chunks.append(chunk)
    with zipfile.ZipFile(io.BytesIO(b''.join(chunks))) as z:
        csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]
        with z.open(csv_name) as f:
            df_raw = pd.read_csv(f, low_memory=False)
    df_raw.to_csv(OPL_PATH, index=False)
else:
    df_raw = pd.read_csv(OPL_PATH, low_memory=False)
    print(f'Loaded — {len(df_raw):,} records')

In [ ]:
df = df_raw[['Name','Sex','Best3SquatKg','Equipment']].copy()
df = df[df['Best3SquatKg'] > 0].dropna(subset=['Best3SquatKg','Sex','Equipment'])
df['squat_lbs'] = df['Best3SquatKg'] * 2.20462
df = df[df['Equipment'].isin(['Raw','Wraps'])]
df = df[df['Sex'].isin(['M','F'])]
df = df[(df['squat_lbs'] >= 45) & (df['squat_lbs'] <= 950)]

# Deduplicate: keep each lifter's best raw squat only.
# The same person can appear dozens of times across meets. Prolific competitors
# skew stronger (they keep competing because they're good), which oversamples
# the right tail and inflates sigma if left in.
n_before = len(df)
df = df.groupby(['Name','Sex'], as_index=False)['squat_lbs'].max()
n_after = len(df)
print(f'Records before dedup: {n_before:,}')
print(f'Unique lifters after:  {n_after:,}  ({n_before/n_after:.1f}x avg entries per lifter)')

male_opl   = df[df['Sex']=='M']['squat_lbs'].values
female_opl = df[df['Sex']=='F']['squat_lbs'].values

print(f'\nClean unique lifters:  Male {len(male_opl):,}  |  Female {len(female_opl):,}')
print(f'Male   — median {np.median(male_opl):.1f} lbs  |  mean {np.mean(male_opl):.1f} lbs')
print(f'Female — median {np.median(female_opl):.1f} lbs  |  mean {np.mean(female_opl):.1f} lbs')


In [ ]:
m_shape, _, m_scale = _lognorm.fit(male_opl,   floc=0)
f_shape, _, f_scale = _lognorm.fit(female_opl, floc=0)
print(f'Male   σ={m_shape:.4f}  fit_median={m_scale:.1f} lbs')
print(f'Female σ={f_shape:.4f}  fit_median={f_scale:.1f} lbs')

x = np.linspace(45, 850, 600)
fig, axes = make_fig(13, 5, cols=2)
for ax, vals, sh, sc, color, label in [
    (axes[0], male_opl,   m_shape, m_scale, MALE, 'Male'),
    (axes[1], female_opl, f_shape, f_scale, FEM,  'Female'),
]:
    ax.hist(vals, bins=150, density=True, color=color, alpha=0.35, edgecolor='none')
    ax.plot(x, _lognorm.pdf(x, sh, scale=sc), color=WHITE, lw=2.2, label='Log-normal fit')
    ax.axvline(np.median(vals), color=color, lw=1.4, ls='--', alpha=0.8,
               label=f'Median {np.median(vals):.0f} lbs')
    ax.set_title(f'{label} — Raw OPL Squat', color=WHITE, fontsize=12, fontweight='bold')
    ax.set_xlabel('Squat (lbs)'); ax.set_ylabel('Density')
    ax.set_xlim(45, 850)
    ax.legend(fontsize=9, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
    for t, c in zip(MILESTONES, M_COLORS):
        ax.axvline(t, color=c, lw=1.2, ls=':', alpha=0.7)
fig.suptitle('Log-Normal Fit — OPL Raw Squat Data', color=WHITE, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('01_opl_fit.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 🏋️ Chapter 2 — Two Pools of Squatters

The bench press is almost purely a gym movement. The squat is not.

**Pool A — Gym barbell squatters:** people who specifically train the back squat in a gym setting. Same methodology as bench model: keep OPL shape (sigma), shift median down to normative gym-goer data.

**Pool B — Non-gym strength athletes:** people who have developed real leg strength through:
- Military service (20M+ active duty globally; many programs include loaded carries, ruck marching, sprinting — compound leg work)
- Combat sports — wrestling, judo, boxing, MMA, sambo (~30–50M serious practitioners)
- Rugby union/league (~8M registered players), American/Canadian football (~5M)
- Serious team sport athletes with structured S&C (professional/semi-pro soccer, basketball)

**Key adjustment for Pool B:** a 70% barbell efficiency factor. Someone who has never specifically trained the back squat movement (bracing, bar path, depth under load) can't fully express their leg strength on day one. Conservative: effective 1RM ≈ 70% of their true strength potential. This is already baked into Pool B's median — they're calibrated to what they could actually lift on a barbell, not their theoretical leg strength ceiling.

Pool B is **heavily male-dominated** (~75% male) because military and combat sports participation skews heavily male.

In [ ]:
# Visualize the two pool distributions vs OPL competitors
fig, ax = make_fig(13, 5)
x = np.linspace(20, 850, 800)

# Pool A: gym squatters (North America baseline)
gym_m_med, gym_f_med = 205.0, 115.0
gym_sig_m, gym_sig_f = 0.33, 0.345

# Pool B: non-gym strength athletes (barbell-efficiency-adjusted medians)
# Male: ~175 lbs effective squat (strong from sport, but not squat-specific trained)
# Female: ~105 lbs effective squat
ng_m_med, ng_f_med = 175.0, 105.0
ng_sig_m, ng_sig_f = 0.35, 0.36   # slightly wider — less systematic training

curves = [
    (np.median(male_opl),   m_shape,   MALE,          'Male — OPL competitors',         '-',  3.0),
    (gym_m_med,             gym_sig_m, POOL_A_COLOR,  'Male — Pool A (gym squatter)',   '--', 2.0),
    (ng_m_med,              ng_sig_m,  POOL_B_COLOR,  'Male — Pool B (non-gym athlete)',':', 2.0),
    (np.median(female_opl), f_shape,   FEM,           'Female — OPL competitors',       '-',  2.0),
    (gym_f_med,             gym_sig_f, '#c8a0d8',     'Female — Pool A',                '--', 1.5),
    (ng_f_med,              ng_sig_f,  '#ffcc80',     'Female — Pool B',                ':',  1.5),
]
for med, sigma, color, label, ls, lw in curves:
    ax.plot(x, _lognorm.pdf(x, sigma, scale=med),
            color=color, lw=lw, ls=ls, label=label, alpha=0.9)

for t, c in zip(MILESTONES, M_COLORS):
    ax.axvline(t, color=c, lw=1.2, ls=':', alpha=0.6)
    ax.text(t+4, ax.get_ylim()[1]*0.03, f'{t}', color=c, fontsize=9)

ax.set_xlabel('Squat (lbs)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Three Populations: OPL Competitors | Gym Squatters | Non-Gym Athletes',
             color=WHITE, fontsize=13, fontweight='bold')
ax.legend(fontsize=9, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID, ncol=2)
ax.set_xlim(20, 750)
plt.tight_layout()
plt.savefig('02_pools.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 🌍 Chapter 3 — Regional Parameters

**Pool A** follows the same 10-region structure as the bench model.  
**Pool B** (non-gym athletes) is defined separately — sizes derived from:
- IISS Military Balance 2023 (active duty by region)
- World combat sports federation membership estimates
- World Rugby, FIFA, FIBA registered serious-level athlete estimates

Pool B is **not** dependent on gym access — military and combat sports exist at roughly the same absolute numbers regardless of commercial gym penetration.

In [ ]:
# ── POOL A: Gym barbell squatters ───────────────────────────────────────────
# MSquatFrac: fraction of resistance trainers who back squat (lower than bench;
# many use Smith machine, leg press, or skip legs)
# FSquatFrac: women's squat rack adoption higher than bench, but still < male rate
#
# Columns: name, pop_16_70(M), lift_mode, lift_lo, lift_hi,
#          MSquatFrac, FSquatFrac,
#          male_med, male_lo, male_hi, female_med, female_lo, female_hi

REGIONS_A = [
    ('North America',         210, 0.22, 0.17, 0.28, 0.62, 0.38, 205, 185, 225, 115,  98, 132),
    ('Europe',                450, 0.15, 0.10, 0.20, 0.58, 0.34, 190, 172, 208, 108,  92, 124),
    ('Russia / C. Asia',      165, 0.10, 0.07, 0.14, 0.60, 0.34, 185, 167, 203, 105,  89, 121),
    ('East Asia',             950, 0.07, 0.04, 0.11, 0.50, 0.30, 160, 144, 176,  90,  76, 104),
    ('Latin America',         380, 0.08, 0.05, 0.12, 0.58, 0.36, 165, 149, 181,  93,  79, 107),
    ('Middle East / N. Af.',  310, 0.05, 0.03, 0.08, 0.54, 0.28, 165, 149, 181,  88,  74, 102),
    ('South Asia',           1080, 0.025,0.015,0.040,0.50, 0.28, 140, 125, 155,  77,  65,  89),
    ('SE Asia',               500, 0.035,0.020,0.055,0.52, 0.30, 148, 133, 163,  83,  70,  96),
    ('Sub-Saharan Africa',    610, 0.015,0.008,0.025,0.50, 0.28, 148, 133, 163,  83,  70,  96),
    ('Oceania',                19, 0.20, 0.16, 0.25, 0.60, 0.36, 198, 178, 218, 112,  95, 129),
]

cols_a = ['Region','Pop_M','LiftMode','LiftLo','LiftHi','MSquatFrac','FSquatFrac',
          'MMed','MMedLo','MMedHi','FMed','FMedLo','FMedHi']
df_a = pd.DataFrame(REGIONS_A, columns=cols_a)
df_a['Pop_M'] *= 1e6

df_a['EffSF'] = FRAC_MALE * df_a['MSquatFrac'] + (1-FRAC_MALE) * df_a['FSquatFrac']
df_a['Squatters_M'] = df_a['Pop_M'] * df_a['LiftMode'] * df_a['EffSF'] / 1e6
print(f'Pool A — total gym barbell squatters: {df_a["Squatters_M"].sum():.0f}M')

In [ ]:
# ── POOL B: Non-gym strength athletes ───────────────────────────────────────
#
# Population sources:
#   Military: IISS Military Balance 2023, active duty only
#   Combat sports: FILA/UWW (wrestling), IJF (judo), WBC/WBA/IBF (boxing),
#                  ONE Championship, ADCC, UFC pipeline estimates
#   Rugby: World Rugby 2022 census
#   American/Canadian football: NFL/CFL pipeline + college
#   Team sports S&C: serious-level athletes with structured strength programs
#
# MaleFrac: fraction of Pool B that is male (~0.72-0.80; military+combat sports heavily male)
#
# Medians already incorporate 70% barbell efficiency factor:
#   These athletes have real leg strength but have never specifically trained the
#   back squat. On day 1 of squatting they can't fully express their leg strength —
#   bracing, bar path, and depth under load all take reps to learn.
#   70% efficiency is conservative and intentionally cautious.
#
# Columns: name, total_pool_B(M), male_frac,
#          male_med, male_lo, male_hi, female_med, female_lo, female_hi

REGIONS_B = [
    # name                   pool_M  mfrac  mm   mlo  mhi  fm  flo  fhi
    ('North America',          10.0, 0.74, 185, 165, 205, 112,  95, 129),
    ('Europe',                 15.0, 0.73, 175, 157, 193, 107,  90, 124),
    ('Russia / C. Asia',        8.0, 0.78, 172, 154, 190, 103,  87, 119),
    ('East Asia',              15.0, 0.79, 148, 132, 164,  90,  76, 104),
    ('Latin America',           8.0, 0.75, 152, 136, 168,  93,  79, 107),
    ('Middle East / N. Af.',    6.0, 0.82, 152, 136, 168,  88,  74, 102),
    ('South Asia',              5.0, 0.80, 130, 115, 145,  80,  67,  93),
    ('SE Asia',                 5.0, 0.80, 137, 122, 152,  83,  70,  96),
    ('Sub-Saharan Africa',      4.0, 0.79, 137, 122, 152,  83,  70,  96),
    ('Oceania',                 2.0, 0.71, 180, 161, 199, 108,  91, 125),
]

cols_b = ['Region','PoolB_M','MaleFrac',
          'MMed','MMedLo','MMedHi','FMed','FMedLo','FMedHi']
df_b = pd.DataFrame(REGIONS_B, columns=cols_b)
df_b['PoolB_M'] *= 1e6

# ── Overlap correction: remove Pool B members who already barbell squat ──────
# Some fraction of military, combat sports, and rugby/football athletes ALSO
# train in a gym and do barbell squats — they are already captured in Pool A
# at a HIGHER strength level. Counting them in Pool B too would double-count.
#
# The people who overlap are not lost — their squatting ability already shows
# up in Pool A. We only remove them from Pool B.
#
# Overlap fraction estimates (% of Pool B who also do structured barbell work):
#   North America 45%: US/CA military has base gyms; college football players
#     squat as part of S&C; MMA gyms increasingly have barbells
#   Europe 40%: similar gym culture; rugby S&C is heavily barbell-based
#   Russia/C.Asia 32%: sambo/judo athletes increasingly use barbell programming
#   East Asia 22%: military PT is bodyweight + cardio dominant
#   Latin America 25%: growing but gym access still limited among combat athletes
#   Middle East/N.Af 18%: military PT mostly bodyweight; barbell gyms sparse
#   South Asia 10%: very limited gym access even in military
#   SE Asia 12%: similar to South Asia
#   Sub-Saharan Africa 8%: lowest gym infrastructure
#   Oceania 42%: NRL/AFL S&C programs are barbell-heavy; military gyms well-equipped

OVERLAP_FRAC = [0.45, 0.40, 0.32, 0.22, 0.25, 0.18, 0.10, 0.12, 0.08, 0.42]
df_b['OverlapFrac'] = OVERLAP_FRAC
df_b['PoolB_Net']   = df_b['PoolB_M'] * (1 - df_b['OverlapFrac'])

total_b_raw = df_b['PoolB_M'].sum()  / 1e6
total_b_net = df_b['PoolB_Net'].sum() / 1e6
print(f'Pool B — raw:        {total_b_raw:.0f}M athletes (military + combat + rugby/football)')
print(f'Pool B — overlap:   -{total_b_raw - total_b_net:.0f}M already counted in Pool A')
print(f'Pool B — net:        {total_b_net:.0f}M truly non-gym squatters added')
print(f'Male fraction (net): {(df_b["PoolB_Net"]*df_b["MaleFrac"]).sum()/df_b["PoolB_Net"].sum():.2f}')


---
## 🎲 Chapter 4 — Monte Carlo: All Three Pools

**Three pools, no double-counting:**

| Pool | Who | Already counted? |
|---|---|---|
| A | Gym barbell squatters (16–70) | No overlap with B or C |
| B | Non-gym strength athletes (16–70), minus those who also gym | Overlap-corrected |
| C | Youth strong athletes (14–15) | Separate age bracket, not in A or B |

**Pool C — Youth athletes (age 14–15):**  
The OPL data has no age floor — teenage competitors shape the fitted distribution — but the *population base* currently starts at 16. Some 14–15-year-olds genuinely can squat 315+: elite teenage powerlifters, HS freshman/sophomore football players in serious S&C programs, young wrestlers.

Honest calibration:  
- World population 14–15: ~270M  
- Fraction doing serious strength training: ~0.4–0.8% (far below adult gym rates; most 14-year-olds don't barbell train)  
- Male squat median at 14–15: ~135 lbs (puberty, bodyweight ~130–155 lbs, limited training age)  
- Sigma: wider (0.38–0.48) — more spread since training age varies enormously  
- Net effect: **real but small** — P(X ≥ 315) at median 135 lbs is deep in the tail  

This is added for completeness and honesty. It won't move the headline number meaningfully.

In [ ]:
N_SIM = 100_000
R     = len(df_a)
rng   = np.random.default_rng(42)

pop_a   = df_a['Pop_M'].values
msf_v   = df_a['MSquatFrac'].values
fsf_v   = df_a['FSquatFrac'].values
pop_b   = df_b['PoolB_Net'].values    # overlap-corrected
mfrac_b = df_b['MaleFrac'].values

# Pool A samples
lift_s   = rng.triangular(df_a['LiftLo'].values, df_a['LiftMode'].values,
                           df_a['LiftHi'].values, size=(N_SIM, R))
msf_s    = rng.uniform(msf_v * 0.85, msf_v * 1.15, size=(N_SIM, R))
fsf_s    = rng.uniform(fsf_v * 0.85, fsf_v * 1.15, size=(N_SIM, R))
a_mmed_s = rng.uniform(df_a['MMedLo'].values, df_a['MMedHi'].values, size=(N_SIM, R))
a_fmed_s = rng.uniform(df_a['FMedLo'].values, df_a['FMedHi'].values, size=(N_SIM, R))
a_msig_s = rng.uniform(0.28, 0.38, size=(N_SIM, 1))
a_fsig_s = rng.uniform(0.29, 0.40, size=(N_SIM, 1))

# Pool B samples
b_mmed_s  = rng.uniform(df_b['MMedLo'].values, df_b['MMedHi'].values, size=(N_SIM, R))
b_fmed_s  = rng.uniform(df_b['FMedLo'].values, df_b['FMedHi'].values, size=(N_SIM, R))
b_msig_s  = rng.uniform(0.32, 0.40, size=(N_SIM, 1))
b_fsig_s  = rng.uniform(0.33, 0.42, size=(N_SIM, 1))
b_scale_s = rng.uniform(0.75, 1.25, size=(N_SIM, R))

# ── Pool C: Youth strong athletes (age 14–15) ────────────────────────────────
# World 14-15 population: ~270M total, ~136M male, ~134M female
# Fraction doing serious barbell strength training: Uniform(0.004, 0.008)
#   - Developed regions: HS football (US ~1.1M), powerlifting, wrestling
#   - Globally sparse: most 14-year-olds worldwide don't have barbell access
# Male squat median: Uniform(125, 145) lbs  — puberty, low BW, short training age
# Female squat median: Uniform(75, 95) lbs
# Sigma: wider Uniform(0.38, 0.48) — huge variation in training age at this age
# Male fraction of serious youth strength athletes: ~0.78 (combat sports + football)
#
# These kids are in the OPL tail already (OPL has no age floor), but the
# population *base* for Pool A/B starts at 16, so they need a separate term.

YOUTH_POP_M  = 136_000_000   # male 14-15 worldwide
YOUTH_POP_F  = 134_000_000   # female 14-15 worldwide

c_rate_s  = rng.uniform(0.004, 0.008, size=(N_SIM,))     # fraction doing serious barbell work
c_mmed_s  = rng.uniform(125, 145, size=(N_SIM,))         # male squat median
c_fmed_s  = rng.uniform(75,   95, size=(N_SIM,))         # female squat median
c_msig_s  = rng.uniform(0.38, 0.48, size=(N_SIM,))       # wider spread
c_fsig_s  = rng.uniform(0.39, 0.50, size=(N_SIM,))

c_mcnt = YOUTH_POP_M * c_rate_s   # (N_SIM,)
c_fcnt = YOUTH_POP_F * c_rate_s * 0.35   # female youth strength athletes ~35% of male rate

a_mcnt = pop_a * lift_s * msf_s * FRAC_MALE
a_fcnt = pop_a * lift_s * fsf_s * (1 - FRAC_MALE)
b_mcnt = pop_b * mfrac_b       * b_scale_s
b_fcnt = pop_b * (1-mfrac_b)   * b_scale_s

mc_results_a = {}
mc_results_b = {}
mc_results_c = {}
mc_results   = {}

for target in MILESTONES:
    pm_a = stats.norm.sf((np.log(target) - np.log(a_mmed_s)) / a_msig_s)
    pf_a = stats.norm.sf((np.log(target) - np.log(a_fmed_s)) / a_fsig_s)
    pool_a = (a_mcnt * pm_a + a_fcnt * pf_a).sum(axis=1) / 1e6
    mc_results_a[target] = pool_a

    pm_b = stats.norm.sf((np.log(target) - np.log(b_mmed_s)) / b_msig_s)
    pf_b = stats.norm.sf((np.log(target) - np.log(b_fmed_s)) / b_fsig_s)
    pool_b = (b_mcnt * pm_b + b_fcnt * pf_b).sum(axis=1) / 1e6
    mc_results_b[target] = pool_b

    pm_c = stats.norm.sf((np.log(target) - np.log(c_mmed_s)) / c_msig_s)
    pf_c = stats.norm.sf((np.log(target) - np.log(c_fmed_s)) / c_fsig_s)
    pool_c = (c_mcnt * pm_c + c_fcnt * pf_c) / 1e6
    mc_results_c[target] = pool_c

    mc_results[target] = pool_a + pool_b + pool_c

print(f'Simulation complete — {N_SIM:,} trials\n')
print(f'{"Target":<8} {"Pool A":>8} {"Pool B":>8} {"Pool C":>10} {"Combined":>10} {"% World":>10}')
print('-'*60)
for t in MILESTONES:
    va  = np.median(mc_results_a[t])
    vb  = np.median(mc_results_b[t])
    vc  = np.median(mc_results_c[t]) * 1000   # show in thousands for C
    vtot = np.median(mc_results[t])
    lo  = np.percentile(mc_results[t], 2.5)
    hi  = np.percentile(mc_results[t], 97.5)
    pct = vtot / (WORLD_POP / 1e6) * 100
    print(f'{t} lbs  {va:>7.2f}M  {vb:>7.2f}M  {vc:>7.0f}k  {vtot:>9.2f}M  {pct:.5f}%')
    print(f'         95% CI: [{lo:.2f}M – {hi:.2f}M]')
print('\nPool C is in thousands (k) — real but small relative to A+B.')


---
## 👥 Chapter 5 — Denominator Analysis

The denominator you choose changes the story dramatically:

| Denominator | Who's in it | Why use it |
|---|---|---|
| All 8.2B humans | Every living person | Most honest: the question is 'how many people on Earth' |
| Able adults 16–70 | Excluded: true children, elderly 70+ | Removes people for whom it's physically impossible |
| Men 16–70 only | ~2.37B | Most relevant if asking 'how rare among men' |
| Women 16–70 only | ~2.30B | Same for women — paints a completely different picture |

In [ ]:
mc_male   = {}
mc_female = {}

for target in MILESTONES:
    pm_a = stats.norm.sf((np.log(target) - np.log(a_mmed_s)) / a_msig_s)
    pf_a = stats.norm.sf((np.log(target) - np.log(a_fmed_s)) / a_fsig_s)
    pm_b = stats.norm.sf((np.log(target) - np.log(b_mmed_s)) / b_msig_s)
    pf_b = stats.norm.sf((np.log(target) - np.log(b_fmed_s)) / b_fsig_s)
    pm_c = stats.norm.sf((np.log(target) - np.log(c_mmed_s)) / c_msig_s)
    pf_c = stats.norm.sf((np.log(target) - np.log(c_fmed_s)) / c_fsig_s)

    mc_male[target]   = (a_mcnt*pm_a + b_mcnt*pm_b).sum(axis=1)/1e6 + c_mcnt*pm_c/1e6
    mc_female[target] = (a_fcnt*pf_a + b_fcnt*pf_b).sum(axis=1)/1e6 + c_fcnt*pf_c/1e6

# ── Gym-goer denominator ─────────────────────────────────────────────────────
# Total resistance trainers worldwide (anyone who resistance trains, regardless
# of whether they barbell squat). This is the denominator most gym people relate
# to — "of people who actually lift, how many can hit this?"
#
# = sum(regional_pop × lift_rate) — using the MC lift_s draws for uncertainty
gym_goers_s = (pop_a * lift_s).sum(axis=1) / 1e6   # (N_SIM,) in millions
POP_GYM     = np.median(gym_goers_s)                 # point estimate ~300M

POP_ABLE_1470 = POP_ABLE_1670 + 270_000_000

print(f'Total resistance trainers (median MC): {POP_GYM:.0f}M')
print(f'  95% CI: [{np.percentile(gym_goers_s,2.5):.0f}M – {np.percentile(gym_goers_s,97.5):.0f}M]')
print()
print('='*78)
print(f'{"":10} {"All 8.2B":>10} {"Able 16-70":>12} {"Men 16-70":>12} {"Gym-goers":>12}')
print('='*78)
for t in MILESTONES:
    vc = np.median(mc_results[t])
    vm = np.median(mc_male[t])
    vf = np.median(mc_female[t])
    vc_youth = np.median(mc_results_c[t])
    pct_gym = vc / POP_GYM * 100
    print(f'\n{t} lbs  ≈{vc:.2f}M total  (youth 14-15 add {vc_youth*1000:.0f}k)')
    print(f'  % of all humans:   {vc/(WORLD_POP/1e6)*100:.5f}%   1 in {int(WORLD_POP/(vc*1e6)):,}')
    print(f'  % of able 16-70:   {vc/(POP_ABLE_1670/1e6)*100:.4f}%   1 in {int(POP_ABLE_1670/(vc*1e6)):,}')
    print(f'  % of men 16-70:    {vm/(POP_MEN_1670/1e6)*100:.4f}%   1 in {int(POP_MEN_1670/(vm*1e6)):,}')
    print(f'  % of gym-goers:    {pct_gym:.2f}%   1 in {int(POP_GYM/vc):,} gym-goers')
    print(f'  % of women 16-70:  {vf/(POP_WOMEN_1670/1e6)*100:.5f}%')


In [ ]:
# ── Pool A vs B stacked bar ────────────────────────────────────────────────
fig, ax = make_fig(10, 5)

x    = np.arange(len(MILESTONES))
va   = [np.median(mc_results_a[t]) for t in MILESTONES]
vb   = [np.median(mc_results_b[t]) for t in MILESTONES]
vc   = [np.median(mc_results[t])   for t in MILESTONES]

bars_a = ax.bar(x, va, color=POOL_A_COLOR, alpha=0.85, label='Pool A — gym squatters')
bars_b = ax.bar(x, vb, bottom=va, color=POOL_B_COLOR, alpha=0.85, label='Pool B — non-gym athletes')

for i, (a, b, c) in enumerate(zip(va, vb, vc)):
    ax.text(i, c + 0.05, f'{c:.2f}M total', ha='center', color=WHITE, fontsize=11, fontweight='bold')
    ax.text(i, a/2,     f'{a:.2f}M', ha='center', color=BG,   fontsize=10)
    ax.text(i, a + b/2, f'{b:.2f}M', ha='center', color=BG,   fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(M_TAGS, color=WHITE, fontsize=12)
ax.set_ylabel('People worldwide (millions)', fontsize=11)
ax.set_title('Pool A (gym) vs Pool B (non-gym athletes) — Both Pools Combined',
             color=WHITE, fontsize=13, fontweight='bold')
ax.legend(fontsize=10, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
plt.tight_layout()
plt.savefig('03_pool_split.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ── Male vs Female split ──────────────────────────────────────────────────
fig, ax = make_fig(10, 5)
x  = np.arange(len(MILESTONES))
vm = [np.median(mc_male[t])   for t in MILESTONES]
vf = [np.median(mc_female[t]) for t in MILESTONES]

w = 0.35
bm = ax.bar(x - w/2, vm, w, color=MALE, alpha=0.85, label='Male')
bf = ax.bar(x + w/2, vf, w, color=FEM,  alpha=0.85, label='Female')

for i, (m, f) in enumerate(zip(vm, vf)):
    pct_m = m / (POP_MEN_1670/1e6) * 100
    pct_f = f / (POP_WOMEN_1670/1e6) * 100
    ax.text(i-w/2, m+0.03, f'{m:.2f}M\n({pct_m:.3f}%)',
            ha='center', va='bottom', color=MALE, fontsize=9, fontweight='bold')
    ax.text(i+w/2, f+0.03, f'{f:.2f}M\n({pct_f:.4f}%)',
            ha='center', va='bottom', color=FEM,  fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(M_TAGS, color=WHITE, fontsize=12)
ax.set_ylabel('People worldwide (millions)', fontsize=11)
ax.set_title('Male vs Female — Who Can Actually Hit These Numbers?',
             color=WHITE, fontsize=13, fontweight='bold')
ax.legend(fontsize=11, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
plt.tight_layout()
plt.savefig('04_male_female_split.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ── Full drop-off curve: 100-600 lbs ────────────────────────────────────────
X_TARGETS = np.linspace(100, 600, 200)
N_CURVE   = 10_000
rng3      = np.random.default_rng(7)

lc_a   = rng3.triangular(df_a['LiftLo'].values, df_a['LiftMode'].values,
                          df_a['LiftHi'].values, size=(N_CURVE, R))
mc_as  = rng3.uniform(msf_v*0.85, msf_v*1.15, size=(N_CURVE, R))
fc_as  = rng3.uniform(fsf_v*0.85, fsf_v*1.15, size=(N_CURVE, R))
am_mc  = rng3.uniform(df_a['MMedLo'].values, df_a['MMedHi'].values, size=(N_CURVE, R))
af_mc  = rng3.uniform(df_a['FMedLo'].values, df_a['FMedHi'].values, size=(N_CURVE, R))
ams_c  = rng3.uniform(0.28, 0.38, size=(N_CURVE,1,1))
afs_c  = rng3.uniform(0.29, 0.40, size=(N_CURVE,1,1))

bsc    = rng3.uniform(0.75, 1.25, size=(N_CURVE, R))
bm_mc  = rng3.uniform(df_b['MMedLo'].values, df_b['MMedHi'].values, size=(N_CURVE, R))
bf_mc  = rng3.uniform(df_b['FMedLo'].values, df_b['FMedHi'].values, size=(N_CURVE, R))
bms_c  = rng3.uniform(0.32, 0.40, size=(N_CURVE,1,1))
bfs_c  = rng3.uniform(0.33, 0.42, size=(N_CURVE,1,1))

# Use overlap-corrected pool size (PoolB_Net)
a_mc3  = (pop_a * lc_a * mc_as * FRAC_MALE)[:,:,None]
a_fc3  = (pop_a * lc_a * fc_as * (1-FRAC_MALE))[:,:,None]
b_mc3  = (pop_b * mfrac_b * bsc)[:,:,None]    # pop_b is already PoolB_Net
b_fc3  = (pop_b * (1-mfrac_b) * bsc)[:,:,None]

ln_x   = np.log(X_TARGETS)[None,None,:]

curve_a = (a_mc3 * stats.norm.sf((ln_x - np.log(am_mc[:,:,None])) / ams_c) +
           a_fc3 * stats.norm.sf((ln_x - np.log(af_mc[:,:,None])) / afs_c)).sum(axis=1)/1e6
curve_b = (b_mc3 * stats.norm.sf((ln_x - np.log(bm_mc[:,:,None])) / bms_c) +
           b_fc3 * stats.norm.sf((ln_x - np.log(bf_mc[:,:,None])) / bfs_c)).sum(axis=1)/1e6
curve_t = curve_a + curve_b

c_med  = np.median(curve_t, axis=0)
c_lo   = np.percentile(curve_t, 2.5,  axis=0)
c_hi   = np.percentile(curve_t, 97.5, axis=0)
ca_med = np.median(curve_a, axis=0)

fig, ax = make_fig(13, 6)
ax.fill_between(X_TARGETS, c_lo, c_hi, color=WHITE, alpha=0.10, label='95% CI (combined)')
ax.plot(X_TARGETS, c_med,  color=WHITE,        lw=2.5, label='Combined (gym + non-gym athletes)')
ax.plot(X_TARGETS, ca_med, color=POOL_A_COLOR, lw=1.5, ls='--', alpha=0.7, label='Pool A only (gym)')

for t, color, lbl in zip(MILESTONES, M_COLORS, M_LABELS):
    idx = int(np.argmin(np.abs(X_TARGETS - t)))
    ax.axvline(t, color=color, lw=1.5, ls=':', alpha=0.7)
    ax.scatter([t], [c_med[idx]], color=color, s=90, zorder=5)
    ax.text(t+5, c_med[idx]*1.4, f'{lbl}\n{c_med[idx]:.2f}M',
            color=color, fontsize=9.5, va='bottom')

ax.set_yscale('log')
ax.set_xlabel('Squat (lbs)', fontsize=12)
ax.set_ylabel('People worldwide (millions, log)', fontsize=12)
ax.set_title('Global Squat Capability — Full Drop-Off Curve\n(overlap-corrected: gym + non-gym athletes)',
             color=WHITE, fontsize=13, fontweight='bold')
ax.legend(fontsize=10, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.3g}M'))
plt.tight_layout()
plt.savefig('05_dropoff.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


---
## 🎬 IG Reel Slides — 9:16 Portrait

In [ ]:
# ── Weighted global medians for the distribution illustration ────────────────
_mw = df_a['Pop_M'] * df_a['LiftMode'] * df_a['MSquatFrac']
_fw = df_a['Pop_M'] * df_a['LiftMode'] * df_a['FSquatFrac']
_gm_med = float(np.average(df_a['MMed'], weights=_mw))
_gf_med = float(np.average(df_a['FMed'], weights=_fw))
_m_sig, _f_sig = 0.33, 0.345

def _reel_fig():
    return plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)

def _chart_axes(fig, bottom, height):
    ax = fig.add_axes([0.06, bottom, 0.90, height])
    _style(ax)
    return ax

def draw_reveal_chart(target, color, fig, chart_bottom=0.38, chart_height=0.54):
    """
    Distribution curve with shaded tail.
    Below the chart: M and F stats in two equal columns — both audiences see their number.
    Returns (med_total, med_male, med_female, pct_gym_male, pct_gym_female).
    """
    sims = mc_results[target]
    med  = np.median(sims)
    lo, hi = np.percentile(sims, [2.5, 97.5])
    vm   = np.median(mc_male[target])
    vf   = np.median(mc_female[target])

    pct_gym_m = vm / (POP_GYM * FRAC_MALE) * 100        # % of male gym-goers
    pct_gym_f = vf / (POP_GYM * (1-FRAC_MALE)) * 100    # % of female gym-goers
    in_x_m    = int(POP_MEN_1670   / (vm * 1e6))
    in_x_f    = int(POP_WOMEN_1670 / (vf * 1e6))

    # ── Distribution chart ───────────────────────────────────────────────────
    ax = fig.add_axes([0.06, chart_bottom, 0.90, chart_height])
    _style(ax)

    x = np.linspace(20, 750, 1400)
    pdf_m = _lognorm.pdf(x, _m_sig, scale=_gm_med)
    pdf_f = _lognorm.pdf(x, _f_sig, scale=_gf_med)
    y_top = max(pdf_m.max(), pdf_f.max())

    ax.plot(x, pdf_m, color=MALE, lw=2.8, alpha=0.9)
    ax.plot(x, pdf_f, color=FEM,  lw=2.8, alpha=0.9)
    mask = x >= target
    ax.fill_between(x[mask], pdf_m[mask], color=MALE, alpha=0.28)
    ax.fill_between(x[mask], pdf_f[mask], color=FEM,  alpha=0.28)
    ax.axvline(target, color=color, lw=3.5, ls='--', alpha=0.95)
    ax.text(target + 9, y_top * 0.97,
            f'{target} lbs', color=color, fontsize=13, fontweight='black', va='top')

    # Small combined total in chart body
    ax.text(0.04, 0.93, f'≈ {med:.2f}M total worldwide',
            transform=ax.transAxes, ha='left', va='top',
            color=WHITE, fontsize=13, fontweight='bold')
    ax.text(0.04, 0.82, f'95% CI: {lo:.2f}M – {hi:.2f}M',
            transform=ax.transAxes, ha='left', va='top',
            color=GRAY, fontsize=10)

    ax.set_xlim(20, 750)
    ax.set_xlabel('Squat (lbs)', fontsize=11, labelpad=3)

    # ── Side-by-side M / F stat block below chart ────────────────────────────
    # Left column: Men    Right column: Women
    col_y  = chart_bottom - 0.04          # top of stat block
    left_x, right_x = 0.04, 0.53

    # Column headers
    fig.text(left_x,  col_y, '♂  MEN',   ha='left',  va='top',
             color=MALE, fontsize=16, fontweight='black')
    fig.text(right_x, col_y, '♀  WOMEN', ha='left',  va='top',
             color=FEM,  fontsize=16, fontweight='black')

    # Big number
    fig.text(left_x,  col_y-0.055, f'{vm:.2f}M', ha='left', va='top',
             color=MALE, fontsize=26, fontweight='black')
    fig.text(right_x, col_y-0.055, f'{vf:.3f}M', ha='left', va='top',
             color=FEM,  fontsize=26, fontweight='black')

    # % of gym-goers
    fig.text(left_x,  col_y-0.108, f'~{pct_gym_m:.1f}% of male gym-goers',
             ha='left', va='top', color=WHITE, fontsize=11, fontweight='bold')
    fig.text(right_x, col_y-0.108, f'~{pct_gym_f:.2f}% of female gym-goers',
             ha='left', va='top', color=WHITE, fontsize=11, fontweight='bold')

    # 1 in X
    fig.text(left_x,  col_y-0.148, f'1 in every {in_x_m:,} men',
             ha='left', va='top', color=GRAY, fontsize=11)
    fig.text(right_x, col_y-0.148, f'1 in every {in_x_f:,} women',
             ha='left', va='top', color=GRAY, fontsize=11)

    # Divider between columns
    fig.add_axes([0.50, chart_bottom - 0.18, 0.005, 0.17]).set_visible(False)
    line_ax = fig.add_axes([0.499, chart_bottom-0.185, 0.003, 0.175])
    line_ax.set_facecolor(GRID); line_ax.axis('off')

    return med, vm, vf, pct_gym_m, pct_gym_f, in_x_m, in_x_f


In [ ]:
# ── Slide 00 — HOOK ──────────────────────────────────────────────────────────
fig = _reel_fig()
ax  = fig.add_axes([0,0,1,1]); ax.set_facecolor(BG); ax.axis('off'); T = ax.transAxes

ax.axhline(0.95, color=C315, lw=5, xmin=0.06, xmax=0.94)

# Lead with the viewer, not the model
ax.text(0.5, 0.91, 'If you squat', ha='center', va='top',
        color=GRAY, fontsize=28, fontweight='bold', transform=T)
ax.text(0.5, 0.82, '3 plates', ha='center', va='top',
        color=C315, fontsize=66, fontweight='black', transform=T)
ax.text(0.5, 0.72, 'here\'s what that\nactually means.', ha='center', va='top',
        color=WHITE, fontsize=32, fontweight='black', transform=T, linespacing=1.2)

ax.axhline(0.62, color=GRID, lw=1.5, xmin=0.1, xmax=0.9, alpha=0.5)

ax.text(0.5, 0.58, 'I modeled every person on Earth —', ha='center', va='top',
        color=GRAY, fontsize=16, transform=T)
ax.text(0.5, 0.52, 'gym-goers, military, combat athletes,\neven high school football.',
        ha='center', va='top', color=GRAY, fontsize=16, transform=T, linespacing=1.4)
ax.text(0.5, 0.43, '8.2 billion people.', ha='center', va='top',
        color=WHITE, fontsize=20, fontweight='bold', transform=T)

ax.axhline(0.37, color=GRID, lw=1.0, xmin=0.1, xmax=0.9, alpha=0.4)

for i, (m, c, lbl) in enumerate(zip(MILESTONES, M_COLORS, ['3 plates','4 plates','5 plates'])):
    xp = 0.18 + i*0.32
    ax.text(xp, 0.33, f'{m}', ha='center', color=c, fontsize=30,
            fontweight='black', transform=T)
    ax.text(xp, 0.26, lbl, ha='center', color=GRAY, fontsize=13, transform=T)

ax.text(0.5, 0.16, 'The numbers will humble you.', ha='center', va='top',
        color=WHITE, fontsize=18, fontweight='bold', transform=T)
ax.text(0.5, 0.09, 'Stay for 495 — it\'s the wildest one. ↓',
        ha='center', va='top', color=C495, fontsize=15, fontweight='bold', transform=T)
ax.axhline(0.04, color=C495, lw=3, xmin=0.06, xmax=0.94, alpha=0.6)

plt.savefig('reel_00_hook.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print('Slide 00 saved')


In [ ]:
# ── Slide 00b — AUTHORITY (fast, confident, ~5 seconds on screen) ────────────
# Goal: one real number from the data, one contrast, builds trust immediately.
# No methodology lecture — just proof that you know the difference between
# competition lifters and regular people, and corrected for it.

fig = _reel_fig()

# Top section: the data source + OPL median
fig.text(0.5, 0.985, 'THE DATA', ha='center', va='top',
         color=WHITE, fontsize=44, fontweight='black')
fig.text(0.5, 0.935, '3.9 million real competition lifts.',
         ha='center', va='top', color=C315, fontsize=22, fontweight='bold')
fig.text(0.5, 0.890, 'OpenPowerlifting — every sanctioned meet on record.',
         ha='center', va='top', color=GRAY, fontsize=14)

# Small comparison chart — OPL competitor vs gym-goer baseline
ax = fig.add_axes([0.10, 0.50, 0.82, 0.34])
_style(ax)

x_plot = np.linspace(50, 800, 800)
opl_m_med = np.median(male_opl)
opl_f_med = np.median(female_opl)

# Gym-goer baseline (global weighted average from Pool A)
gym_m = _gm_med
gym_f = _gf_med

for med, sigma, color, label, ls, lw in [
    (opl_m_med, m_shape,  MALE,       f'OPL competitor (median {opl_m_med:.0f} lbs)', '-',  2.5),
    (gym_m,     _m_sig,   '#4fc3f755','Gym-goer model (median {:.0f} lbs)'.format(gym_m), '--', 2.0),
    (opl_f_med, f_shape,  FEM,        f'OPL competitor (median {opl_f_med:.0f} lbs)', '-',  2.0),
    (gym_f,     _f_sig,   '#f48fb155','Gym-goer model (median {:.0f} lbs)'.format(gym_f), '--', 1.5),
]:
    ax.plot(x_plot, _lognorm.pdf(x_plot, sigma, scale=med),
            color=color, lw=lw, ls=ls, label=label, alpha=0.9)

ax.set_xlim(50, 750)
ax.set_xlabel('Squat (lbs)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID, ncol=1)
ax.set_title('Competitors vs. real gym-goers — not the same population.',
             color=WHITE, fontsize=11, fontweight='bold')

# Bottom: the key insight in two lines
fig.text(0.5, 0.47,
         f'Average OPL squatter: {opl_m_med:.0f} lbs.',
         ha='center', va='top', color=WHITE, fontsize=18, fontweight='black')
fig.text(0.5, 0.40,
         'Those are people who paid to peak at a meet.\nThis model corrects for that.',
         ha='center', va='top', color=GRAY, fontsize=14, linespacing=1.4)

fig.text(0.5, 0.29,
         'What happens when you apply the real numbers\nto 8.2 billion people?',
         ha='center', va='top', color=WHITE, fontsize=16, fontweight='bold', linespacing=1.4)
fig.text(0.5, 0.19, '↓ Here\'s what I found ↓',
         ha='center', va='top', color=C315, fontsize=17, fontweight='bold')

ax2 = fig.add_axes([0,0,1,0.08]); ax2.set_facecolor(BG); ax2.axis('off')

plt.savefig('reel_00b_authority.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Slide 00b saved — OPL male median: {opl_m_med:.0f} lbs  female: {opl_f_med:.0f} lbs')


In [ ]:
# ── Slide 01 — 315 lbs ───────────────────────────────────────────────────────
fig = _reel_fig()

fig.text(0.5, 0.985, '3 PLATES  —  315 LBS', ha='center', va='top',
         color=C315, fontsize=34, fontweight='black')
fig.text(0.5, 0.943, 'Counting gym-goers, military, combat athletes — everyone.',
         ha='center', va='top', color=GRAY, fontsize=13)

med, vm, vf, pgm, pgf, ixm, ixf = draw_reveal_chart(315, C315, fig,
                                                      chart_bottom=0.40, chart_height=0.50)

# Bottom hook — one line, tight
fig.text(0.5, 0.07, '↓ 4 plates cuts both those numbers by ~80% ↓',
         ha='center', va='top', color=C405, fontsize=14, fontweight='bold')

plt.savefig('reel_01_315.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'315 — total {med:.2f}M  |  men {vm:.2f}M ({pgm:.1f}% gym) 1in{ixm:,}  |  women {vf:.3f}M ({pgf:.2f}% gym) 1in{ixf:,}')


In [ ]:
# ── Slide 02 — 405 lbs ───────────────────────────────────────────────────────
fig = _reel_fig()

fig.text(0.5, 0.985, '4 PLATES  —  405 LBS', ha='center', va='top',
         color=C405, fontsize=34, fontweight='black')
fig.text(0.5, 0.943, 'Most serious lifters never get here.',
         ha='center', va='top', color=GRAY, fontsize=13)

med, vm, vf, pgm, pgf, ixm, ixf = draw_reveal_chart(405, C405, fig,
                                                      chart_bottom=0.40, chart_height=0.50)
med315 = np.median(mc_results[315])
drop   = (1 - med / med315) * 100

fig.text(0.5, 0.07, f'↓ 5 plates is a different planet — stay for it ↓',
         ha='center', va='top', color=C495, fontsize=14, fontweight='bold')

plt.savefig('reel_02_405.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'405 — total {med:.2f}M  |  men {vm:.2f}M ({pgm:.2f}% gym) 1in{ixm:,}  |  women {vf:.4f}M ({pgf:.3f}% gym) 1in{ixf:,}  |  drop from 315: {drop:.0f}%')


In [ ]:
# ── Slide 03 — 495 lbs ───────────────────────────────────────────────────────
fig = _reel_fig()

fig.text(0.5, 0.985, '5 PLATES  —  495 LBS', ha='center', va='top',
         color=C495, fontsize=34, fontweight='black')
fig.text(0.5, 0.943, 'This is where the chart gets uncomfortable.',
         ha='center', va='top', color=GRAY, fontsize=13)

med, vm, vf, pgm, pgf, ixm, ixf = draw_reveal_chart(495, C495, fig,
                                                      chart_bottom=0.40, chart_height=0.50)

fig.text(0.5, 0.07, '↓ here\'s the full drop-off ↓',
         ha='center', va='top', color=WHITE, fontsize=14, fontweight='bold')

plt.savefig('reel_03_495.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'495 — total {med:.2f}M  |  men {vm:.3f}M ({pgm:.3f}% gym) 1in{ixm:,}  |  women {vf:.4f}M ({pgf:.4f}% gym) 1in{ixf:,}')


In [ ]:
# ── Slide 04 — Drop-off curve ────────────────────────────────────────────────
fig = _reel_fig()

fig.text(0.5, 0.985, 'THE DROP-OFF', ha='center', va='top',
         color=WHITE, fontsize=40, fontweight='black')
fig.text(0.5, 0.945, 'Every person on Earth who could possibly squat heavy.',
         ha='center', va='top', color=GRAY, fontsize=13)

# Chart takes up 65% — it IS the slide
ax = fig.add_axes([0.10, 0.22, 0.87, 0.70])
_style(ax)
ax.fill_between(X_TARGETS, c_lo, c_hi, color=WHITE, alpha=0.08, label='95% CI')
ax.plot(X_TARGETS, c_med,  color=WHITE, lw=3.0, label='All people')
ax.plot(X_TARGETS, ca_med, color=POOL_A_COLOR, lw=1.8, ls='--', alpha=0.65, label='Gym only')

for t, color, lbl in zip(MILESTONES, M_COLORS, M_LABELS):
    idx = int(np.argmin(np.abs(X_TARGETS - t)))
    ax.axvline(t, color=color, lw=2.2, ls=':', alpha=0.85)
    ax.scatter([t], [c_med[idx]], color=color, s=110, zorder=5)
    ax.text(t + 5, c_med[idx] * 1.6, f'{lbl}\n{c_med[idx]:.2f}M',
            color=color, fontsize=9.5, va='bottom')

ax.set_yscale('log')
ax.set_xlabel('Squat (lbs)', fontsize=13)
ax.set_ylabel('People worldwide (M, log)', fontsize=12)
ax.legend(fontsize=11, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.3g}M'))

fig.text(0.5, 0.18, 'Each 90 lbs added = ~80% fewer people.',
         ha='center', va='top', color=WHITE, fontsize=15, fontweight='bold')
fig.text(0.5, 0.12, 'Non-gym athletes push the line up at 315.\nBy 495 they\'re gone.',
         ha='center', va='top', color=GRAY, fontsize=13, linespacing=1.4)

plt.savefig('reel_04_dropoff.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print('Slide 04 saved')


In [ ]:
# ── Slide 04b — Squat vs Bench (the expectation-flip slide) ─────────────────
# This is the strongest retention hook: people assume legs > chest means
# squatting 3 plates is MORE common than benching 2 plates. It isn't.
# Stating that expectation and then flipping it keeps people watching.
#
# Bench numbers from bench_world_model (median estimates):
BENCH_EST = {225: 9.27, 315: 1.08, 405: 0.12}
sq_meds   = {t: np.median(mc_results[t]) for t in MILESTONES}

fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
ax  = fig.add_axes([0, 0, 1, 1]); ax.set_facecolor(BG); ax.axis('off'); T = ax.transAxes

ax.axhline(0.94, color=WHITE, lw=4, xmin=0.07, xmax=0.93, alpha=0.4)
ax.text(0.5, 0.97, 'THE EXPECTATION VS REALITY',
        ha='center', va='top', color=WHITE, fontsize=22, fontweight='black', transform=T)

ax.text(0.5, 0.90, '"Legs are stronger than chest —\nsquatting 3 plates should be\nmore common than benching 2 plates."',
        ha='center', va='top', color=GRAY, fontsize=16, fontstyle='italic', transform=T)
ax.text(0.5, 0.77, '— Everyone at the gym',
        ha='center', va='top', color=GRAY, fontsize=14, transform=T)

ax.axhline(0.73, color=GRID, lw=1.5, xmin=0.07, xmax=0.93, alpha=0.6)

# Comparison table
comparisons = [
    ('Bench 225',  BENCH_EST[225], 'Squat 315', sq_meds[315], '2 plates vs 3 plates'),
    ('Bench 315',  BENCH_EST[315], 'Squat 405', sq_meds[405], '3 plates vs 4 plates'),
    ('Bench 405',  BENCH_EST[405], 'Squat 495', sq_meds[495], '4 plates vs 5 plates'),
]

y = 0.70
for blbl, bval, slbl, sval, note in comparisons:
    ratio = bval / sval if sval > 0 else 999
    ax.text(0.5, y, note, ha='center', va='top', color=GRAY, fontsize=13, transform=T)
    y -= 0.048
    ax.text(0.28, y, f'{blbl}', ha='center', va='top',
            color='#888888', fontsize=15, fontweight='bold', transform=T)
    ax.text(0.28, y-0.038, f'{bval:.2f}M', ha='center', va='top',
            color=WHITE, fontsize=18, fontweight='black', transform=T)
    ax.text(0.5, y-0.015, 'vs', ha='center', va='top',
            color=GRAY, fontsize=13, transform=T)
    ax.text(0.72, y, f'{slbl}', ha='center', va='top',
            color=C315, fontsize=15, fontweight='bold', transform=T)
    ax.text(0.72, y-0.038, f'{sval:.2f}M', ha='center', va='top',
            color=C315, fontsize=18, fontweight='black', transform=T)
    ax.text(0.5, y-0.070, f'→ bench is {ratio:.1f}× more common', ha='center', va='top',
            color=C405, fontsize=12, fontweight='bold', transform=T)
    y -= 0.110
    ax.axhline(y+0.01, color=GRID, lw=0.8, xmin=0.07, xmax=0.93, alpha=0.4)

ax.text(0.5, y-0.01,
        'The weight is just higher.\nFewer people reach it — even with\nmore transferable leg strength.',
        ha='center', va='top', color=WHITE, fontsize=14, fontweight='bold', transform=T)

plt.savefig('reel_04b_vs_bench.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print('Slide 04b — Squat vs Bench saved')
for blbl, bval, slbl, sval, note in comparisons:
    print(f'  {note}: bench={bval:.2f}M vs squat={sval:.2f}M ({bval/sval:.1f}x)')


In [ ]:
# ── Slide 05 — Denominator frame (the perspective slide) ────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
ax  = fig.add_axes([0, 0, 1, 1]); ax.set_facecolor(BG); ax.axis('off'); T = ax.transAxes

ax.text(0.5, 0.96, 'DEPENDS WHAT YOU COMPARE TO',
        ha='center', va='top', color=WHITE, fontsize=22, fontweight='black', transform=T)
ax.axhline(0.91, color=GRID, lw=1.5, xmin=0.07, xmax=0.93, alpha=0.5)

rows = [
    ('315 lbs',   C315, [
        ('All 8.2B humans',  f'{np.median(mc_results[315])/(WORLD_POP/1e6)*100:.4f}%',
         f'1 in {int(WORLD_POP/(np.median(mc_results[315])*1e6)):,}'),
        ('Able adults 16-70', f'{np.median(mc_results[315])/(POP_ABLE_1670/1e6)*100:.3f}%',
         f'1 in {int(POP_ABLE_1670/(np.median(mc_results[315])*1e6)):,}'),
        ('Men 16-70 only',   f'{np.median(mc_male[315])/(POP_MEN_1670/1e6)*100:.3f}%',
         f'1 in {int(POP_MEN_1670/(np.median(mc_male[315])*1e6)):,}'),
    ]),
    ('405 lbs',   C405, [
        ('All 8.2B humans',  f'{np.median(mc_results[405])/(WORLD_POP/1e6)*100:.5f}%',
         f'1 in {int(WORLD_POP/(np.median(mc_results[405])*1e6)):,}'),
        ('Able adults 16-70', f'{np.median(mc_results[405])/(POP_ABLE_1670/1e6)*100:.4f}%',
         f'1 in {int(POP_ABLE_1670/(np.median(mc_results[405])*1e6)):,}'),
        ('Men 16-70 only',   f'{np.median(mc_male[405])/(POP_MEN_1670/1e6)*100:.4f}%',
         f'1 in {int(POP_MEN_1670/(np.median(mc_male[405])*1e6)):,}'),
    ]),
    ('495 lbs',   C495, [
        ('All 8.2B humans',  f'{np.median(mc_results[495])/(WORLD_POP/1e6)*100:.6f}%',
         f'1 in {int(WORLD_POP/(np.median(mc_results[495])*1e6)):,}'),
        ('Able adults 16-70', f'{np.median(mc_results[495])/(POP_ABLE_1670/1e6)*100:.5f}%',
         f'1 in {int(POP_ABLE_1670/(np.median(mc_results[495])*1e6)):,}'),
        ('Men 16-70 only',   f'{np.median(mc_male[495])/(POP_MEN_1670/1e6)*100:.5f}%',
         f'1 in {int(POP_MEN_1670/(np.median(mc_male[495])*1e6)):,}'),
    ]),
]

y = 0.88
for milestone_lbl, color, entries in rows:
    ax.text(0.07, y, milestone_lbl, ha='left', va='top',
            color=color, fontsize=20, fontweight='black', transform=T)
    y -= 0.052
    for denom_lbl, pct, one_in in entries:
        ax.text(0.09, y, f'{denom_lbl}:', ha='left', va='top',
                color=GRAY, fontsize=12, transform=T)
        ax.text(0.93, y, f'{pct}  ({one_in})', ha='right', va='top',
                color=WHITE, fontsize=12, fontweight='bold', transform=T)
        y -= 0.044
    ax.axhline(y+0.01, color=GRID, lw=0.8, xmin=0.07, xmax=0.93, alpha=0.5)
    y -= 0.012

ax.text(0.5, y-0.02, 'How you frame it is everything.',
        ha='center', va='top', color=WHITE, fontsize=16, fontweight='bold', transform=T)
ax.text(0.5, y-0.07, 'But the number is always small.', ha='center', va='top',
        color=GRAY, fontsize=14, transform=T)

plt.savefig('reel_05_denominators.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print('Slide 05 — Denominator frame saved')

In [ ]:
# ── Slide 06 — Male vs Female (chart-first) ──────────────────────────────────
fig = _reel_fig()
fig.text(0.5, 0.985, 'MEN vs WOMEN', ha='center', va='top',
         color=WHITE, fontsize=40, fontweight='black')
fig.text(0.5, 0.945, 'Two very different stories.',
         ha='center', va='top', color=GRAY, fontsize=15)

# Chart takes up middle 55%
ax = fig.add_axes([0.08, 0.33, 0.88, 0.57])
_style(ax)
x  = np.arange(len(MILESTONES))
vm = [np.median(mc_male[t])   for t in MILESTONES]
vf = [np.median(mc_female[t]) for t in MILESTONES]
w  = 0.35
ax.bar(x-w/2, vm, w, color=MALE, alpha=0.88, label='Men')
ax.bar(x+w/2, vf, w, color=FEM,  alpha=0.88, label='Women')

for i, (m, f, t) in enumerate(zip(vm, vf, MILESTONES)):
    pm = m / (POP_MEN_1670/1e6) * 100
    pf = f / (POP_WOMEN_1670/1e6) * 100
    pgm = m / POP_GYM * 100
    pgf = f / POP_GYM * 100
    ax.text(i-w/2, m + max(vm)*0.015,
            f'{m:.2f}M\n{pm:.3f}% of men',
            ha='center', va='bottom', color=MALE, fontsize=9.5, fontweight='bold')
    ax.text(i+w/2, f + max(vm)*0.015,
            f'{f:.3f}M\n{pf:.4f}% of women',
            ha='center', va='bottom', color=FEM, fontsize=9.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(M_TAGS, color=WHITE, fontsize=13)
ax.set_ylabel('People worldwide (millions)', fontsize=12)
ax.legend(fontsize=12, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)

ratio = vm[0] / vf[0]
fig.text(0.5, 0.30, f'At 315 lbs, men are {ratio:.0f}× more represented.',
         ha='center', va='top', color=WHITE, fontsize=15, fontweight='bold')
fig.text(0.5, 0.23, 'Not about potential — about who trains\nfor absolute strength and how.',
         ha='center', va='top', color=GRAY, fontsize=14, linespacing=1.4)
fig.text(0.5, 0.11, 'What do you squat? Drop it below 👇',
         ha='center', va='top', color=C315, fontsize=17, fontweight='bold')

plt.savefig('reel_06_male_female.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print('Slide 06 saved')


In [ ]:
# ── Slide 07 — POSITIVE CLOSER ───────────────────────────────────────────────
fig = _reel_fig()
ax  = fig.add_axes([0,0,1,1]); ax.set_facecolor(BG); ax.axis('off'); T = ax.transAxes

ax.axhline(0.95, color=C315, lw=5, xmin=0.06, xmax=0.94)

fig.text(0.5, 0.920, 'These numbers exist', ha='center', va='top',
         color=GRAY, fontsize=22, transform=T)
fig.text(0.5, 0.865, 'to show you what\nyou actually built.',
         ha='center', va='top', color=WHITE, fontsize=34, fontweight='black',
         transform=T, linespacing=1.2)

ax.axhline(0.80, color=GRID, lw=1.2, xmin=0.08, xmax=0.92, alpha=0.5)

for i, (t, c, lbl) in enumerate(zip(MILESTONES, M_COLORS, ['3 plates','4 plates','5 plates'])):
    med     = np.median(mc_results[t])
    pct_gym = med / POP_GYM * 100
    in_x    = int(WORLD_POP / (med * 1e6))
    ybase   = 0.76 - i * 0.155
    ax.text(0.08, ybase, f'{t} lbs  ({lbl})', ha='left', va='top',
            color=c, fontsize=18, fontweight='bold', transform=T)
    ax.text(0.08, ybase - 0.048,
            f'Top ~{pct_gym:.1f}% of gym-goers  •  1 in {in_x:,} people on Earth',
            ha='left', va='top', color=WHITE, fontsize=12, transform=T)
    # axhline uses data coords on the axes, not figure transform — no transform= arg
    ax.axhline(ybase - 0.072, color=GRID, lw=0.7, xmin=0.07, xmax=0.93, alpha=0.4)

ax.text(0.5, 0.30,
        'Billions of people will never squat\nanything close to these numbers.',
        ha='center', va='top', color=GRAY, fontsize=15, transform=T, linespacing=1.4)
ax.text(0.5, 0.20,
        'If you\'re here — you put in the work.',
        ha='center', va='top', color=WHITE, fontsize=20, fontweight='black', transform=T)

ax.axhline(0.13, color=GRID, lw=1.0, xmin=0.08, xmax=0.92, alpha=0.4)

ax.text(0.5, 0.10, 'What\'s your squat? Let\'s see it. 👇',
        ha='center', va='top', color=C315, fontsize=18, fontweight='bold', transform=T)
ax.axhline(0.04, color=C495, lw=3, xmin=0.06, xmax=0.94, alpha=0.6)

plt.savefig('reel_07_closer.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print('Slide 07 — Positive closer saved')


In [ ]:
print('='*65)
print('SQUAT WORLD MODEL v2 — FINAL ESTIMATES')
print('Both pools: gym squatters + non-gym strength athletes')
print('='*65)
for t in MILESTONES:
    vc  = np.median(mc_results[t])
    va  = np.median(mc_results_a[t])
    vb  = np.median(mc_results_b[t])
    vm  = np.median(mc_male[t])
    vf  = np.median(mc_female[t])
    lo  = np.percentile(mc_results[t], 2.5)
    hi  = np.percentile(mc_results[t], 97.5)
    print(f'\n{t} lbs ({t//45} plates):')
    print(f'  Pool A (gym):    {va:.2f}M')
    print(f'  Pool B (athlete):{vb:.2f}M')
    print(f'  Combined:        {vc:.2f}M  [{lo:.2f}M – {hi:.2f}M]')
    print(f'  Male:            {vm:.2f}M  |  Female: {vf:.3f}M')
    print(f'  % all humans:    {vc/(WORLD_POP/1e6)*100:.5f}%')
    print(f'  % able adults:   {vc/(POP_ABLE_1670/1e6)*100:.4f}%')
    print(f'  % of men 16-70:  {vm/(POP_MEN_1670/1e6)*100:.4f}%')
    print(f'  1 in every:      {int(WORLD_POP/(vc*1e6)):,} humans  /  {int(POP_MEN_1670/(vm*1e6)):,} men')